In [0]:
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *

# On récupére le json source qui est sur le bucket jedha
s3_url = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

#Lecture du fichier JSON
df_steam = spark.read.json(s3_url)

# affichage de la structure des données pour avoir les différents noeuds
df_steam.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:

# à partir du prenmier noeud 'data' on crée des colonnes à partir des enfants
df_flat = df_steam.select(
    F.col("id"),
    F.col("data.appid").alias("appid"),
    F.col("data.name").alias("name"),
    F.col("data.publisher").alias("publisher"),
    F.col("data.developer").alias("developer"),
    F.col("data.genre").alias("genre"),
    F.col("data.initialprice").alias("initialprice"),
    F.col("data.price").alias("price"),
    F.col("data.discount").alias("discount"),
    F.col("data.release_date").alias("release_date"),
    F.col("data.required_age").alias("required_age"),
    F.col("data.languages").alias("languages"),
    F.col("data.positive").alias("positive"),
    F.col("data.negative").alias("negative"),
    F.col("data.platforms").alias("platforms") 
)

# aperçu des données
display(df_flat.head(10))

id,appid,name,publisher,developer,genre,initialprice,price,discount,release_date,required_age,languages,positive,negative,platforms
10,10,Counter-Strike,Valve,Valve,Action,999,999,0,2000/11/1,0,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",201215,5199,"List(true, true, true)"
1000000,1000000,ASCENXION,PsychoFlux Entertainment,IndigoBlue Game Studio,"Action, Adventure, Indie",999,999,0,2021/05/14,0,"English, Korean, Simplified Chinese",27,5,"List(false, false, true)"
1000010,1000010,Crown Trick,"Team17, NEXT Studios",NEXT Studios,"Adventure, Indie, RPG, Strategy",1999,599,70,2020/10/16,0,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil",4032,646,"List(false, false, true)"
1000030,1000030,"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,Vertigo Gaming Inc.,"Action, Indie, Simulation, Strategy",1999,1999,0,2020/10/14,0,English,1575,115,"List(false, true, true)"
1000040,1000040,细胞战争,DoubleC Games,DoubleC Games,"Action, Casual, Indie, Simulation",199,199,0,2019/03/30,0,Simplified Chinese,0,1,"List(false, false, true)"
1000080,1000080,Zengeon,2P Games,IndieLeague Studio,"Action, Adventure, Indie, RPG",1999,799,60,2019/06/24,0,"Simplified Chinese, English, Traditional Chinese, Japanese, Korean",1018,462,"List(false, true, true)"
1000100,1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,Starship Studio,七月九日,"Adventure, Indie, RPG, Strategy",1299,1299,0,2019/01/24,0,"Japanese, Simplified Chinese, Traditional Chinese",18,6,"List(false, false, true)"
1000110,1000110,Jumping Master(跳跳大咖),重庆环游者网络科技,重庆环游者网络科技,"Action, Adventure, Casual, Free to Play, Massively Multiplayer",0,0,0,2019/04/8,0,"English, Simplified Chinese, Traditional Chinese",50,34,"List(false, false, true)"
1000130,1000130,Cube Defender,Simon Codrington,Simon Codrington,"Casual, Indie",299,299,0,2019/01/6,0,English,6,0,"List(false, true, true)"
1000280,1000280,Tower of Origin2-Worm's Nest,Villain Role,Villain Role,"Indie, RPG",1399,1399,0,2021/09/9,0,"English, Simplified Chinese, Traditional Chinese",32,12,"List(false, false, true)"


In [0]:
# QUESTION 7 : Les genres les plus représentés

# création de x nouvelles lignes en fonction du split des genres
df_genre_array = df_flat.withColumn("genre_list", F.split(F.col("genre"), ", "))

# Séparation des genres multiples en lignes distinctes via l'extraction des éléments de la liste
df_genre_exploded = df_genre_array.select(
    "name",
    F.explode(F.col("genre_list")).alias("clean_genre")
)

# On sommes dans 'lordre décroissant sans tenir compte des clean_genre null ou vide
df_top_genres = df_genre_exploded.groupBy("clean_genre") \
                                 .count() \
                                 .sort(F.col("count").desc()) \
                                 .filter("clean_genre IS NOT NULL AND clean_genre != ''")

df_top_genres.show(20, truncate=False)

+---------------------+-----+
|clean_genre          |count|
+---------------------+-----+
|Indie                |39681|
|Action               |23759|
|Casual               |22086|
|Adventure            |21431|
|Strategy             |10895|
|Simulation           |10836|
|RPG                  |9534 |
|Early Access         |6145 |
|Free to Play         |3393 |
|Sports               |2666 |
|Racing               |2155 |
|Massively Multiplayer|1460 |
|Utilities            |682  |
|Design & Illustration|406  |
|Animation & Modeling |322  |
|Education            |317  |
|Video Production     |247  |
|Audio Production     |195  |
|Violent              |168  |
|Software Training    |164  |
+---------------------+-----+
only showing top 20 rows


In [0]:
display(df_top_genres)

clean_genre,count
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

![Graphique des genres ](screen_shot/graphique_question7_genres_les_plus_representes.png)

Réponse : <br>
Avec deux fois plus que le deuxième, les jeux indépendants (sans éditeur) sont les grands gagnants de ce palmarès 

In [0]:
# QUESTION 8 : Le meilleur ratio avis positifs/négatifs par genre ?

# déjà défini dans la question 7 en faisant une distinction par 'type' d'avis
df_simple_reviews = df_genre_array.select(
    "positive", 
    "negative", 
    F.explode(F.col("genre_list")).alias("clean_genre")
)

# On somme les avis 
df_genre_scores = df_simple_reviews.groupBy("clean_genre").agg(
    F.sum("positive").alias("positifs"),
    F.sum("negative").alias("negatifs")
)

# Calculet affichage du ratio
df_genre_scores.withColumn(
    "pourcent_satisfaction", 
    F.round((F.col("positifs") / (F.col("positifs") + F.col("negatifs")) * 100), 2)
) \
.filter("positifs > 50000 AND clean_genre != ''") \
.sort(F.col("pourcent_satisfaction").desc()) \
.show(10)

+--------------------+--------+--------+---------------------+
|         clean_genre|positifs|negatifs|pourcent_satisfaction|
+--------------------+--------+--------+---------------------+
|       Photo Editing|  577751|   13745|                97.68|
|Animation & Modeling|  690765|   26392|                96.32|
|Design & Illustra...|  674057|   27007|                96.15|
|           Utilities|  739335|   43503|                94.44|
|               Indie|32531023| 4241234|                88.47|
|    Audio Production|   69118|    9428|                 88.0|
|    Video Production|  111514|   16362|                 87.2|
|              Casual|10034967| 1537296|                86.72|
|          Simulation|15572390| 2400512|                86.64|
|              Racing| 2340353|  383691|                85.91|
+--------------------+--------+--------+---------------------+
only showing top 10 rows


Réponse : <br>
Avec 97.7 pourcents de satisfaction, la retouche photo est en tête suivi de près par la partie 'animation et modeling' .

In [0]:
# QUESTION 9 : Certains éditeurs ont-ils des genres de prédilection ? 

# déjà défini dans la question 7 en faisant une distinction par 'éditeur'
df_pub_genre = df_genre_array.select(
    "publisher",
    F.explode(F.col("genre_list")).alias("clean_genre")
)

# Regrouppement par éditeur et par genre 
df_pub_counts = df_pub_genre.groupBy("publisher", "clean_genre").count()

# on ne tient pas compte des null et des vides pour afficher
df_pub_counts.filter(
    "publisher IS NOT NULL AND trim(publisher) != '' AND clean_genre != ''"
) \
.sort(F.col("count").desc()) \
.show(15, truncate=False)

+--------------------+-----------+-----+
|publisher           |clean_genre|count|
+--------------------+-----------+-----+
|Big Fish Games      |Casual     |418  |
|Big Fish Games      |Adventure  |392  |
|8floor              |Casual     |202  |
|Choice of Games     |RPG        |139  |
|Choice of Games     |Indie      |136  |
|HH-Games            |Casual     |132  |
|Laush Studio        |Indie      |124  |
|Choice of Games     |Adventure  |112  |
|Alawar Entertainment|Casual     |105  |
|Sekai Project       |Casual     |99   |
|Sokpop Collective   |Indie      |97   |
|Slitherine Ltd.     |Strategy   |96   |
|Alawar Entertainment|Adventure  |95   |
|Sekai Project       |Indie      |88   |
|Reforged Group      |Indie      |88   |
+--------------------+-----------+-----+
only showing top 15 rows


![Certains éditeurs ont-ils des genres de prédilection](screen_shot/graphique_question9_editeur_genre_predilection.png)

Réponse : <br>
Oui les éditeurs ont en général un niveau d'expertise professionnelle ciblé sur certains genres, c'est très marqué au niveau du volume pour les éditeurs comme 'Big Fish Game' ou 'Choise of Games'. <br>
A noté des résultats suprenant avec pas moins de 4 éditeurs dans le top 15 qui sont spécialisés dans les jeux hors 'franchise'. 

In [0]:
# QUESTION 10 : Quels sont les genres les plus lucratifs ? 

# On réutilise df_genre_array, on calcule le revenu estimé par jeu et on éclate les genres
df_genre_revenue = df_genre_array.select(
    "positive", "negative", "price",
    F.explode(F.col("genre_list")).alias("clean_genre")
).withColumn(
    "revenu_estime", 
    (F.col("positive") + F.col("negative")) * 30 * F.coalesce(F.col("price").try_cast("float") / 100.0, F.lit(0.0))
)

#agrégation par genre, somme totale
df_genre_lucratif = df_genre_revenue.groupBy("clean_genre") \
    .agg(F.sum("revenu_estime").cast("decimal(20,2)").alias("Revenu_Total_Estime_Euro")) \
    .filter("clean_genre != ''") \
    .sort(F.col("Revenu_Total_Estime_Euro").desc())

df_genre_lucratif.show(10, truncate=False)

+---------------------+------------------------+
|clean_genre          |Revenu_Total_Estime_Euro|
+---------------------+------------------------+
|Action               |31591829265.30          |
|Adventure            |20346799074.00          |
|Indie                |15194681784.90          |
|RPG                  |14771855310.00          |
|Simulation           |10220068667.70          |
|Strategy             |9086575468.50           |
|Massively Multiplayer|3482917740.30           |
|Casual               |3100704366.60           |
|Early Access         |2625555828.00           |
|Sports               |1726796960.40           |
+---------------------+------------------------+
only showing top 10 rows


![Quels sont les genres les plus lucratifs](screen_shot/graphique_question10_genre_lucratif.png)

Réponse : <br>
L'action est Le numéro 1 incontesté avec 1.12 milliard d'avance sur le deuxsième.<br>
A noté que les indépendants ressortent en 3 ème position devant le Role Player Game(RPG), ce qui est un travers de la base Steam. En effet, indie n'est pas un genre en soit mais plutôt un statu. 
